# DrenaVision - treinamento reproduzível no Colab

Este notebook executa o protocolo v2: dados iRodent com caixas derivadas apenas dos keypoints visíveis, expansão do treino e adaptação de domínio. Os dados/métricas v1 permanecem históricos. O alvo corrigido pode cobrir apenas a parte visível do animal e não garante corpo inteiro. Não contém métricas ou outputs pré-preenchidos.

Ative uma GPU em **Ambiente de execução > Alterar tipo de ambiente de execução** antes do treino.

## 1. Obter o repositório

Preencha a URL pública ou uma URL para a qual sua sessão tenha autorização. Não coloque token diretamente no notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
PROJECT = Path("/content/drenavision")

if PROJECT.joinpath("pyproject.toml").is_file():
    print(f"Usando checkout existente: {PROJECT}")
else:
    if not REPO_URL.strip():
        raise ValueError("Preencha REPO_URL com o repositório autorizado.")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(PROJECT)],
        check=True,
    )
os.chdir(PROJECT)
print(subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())


## 2. Instalar o ambiente e verificar a GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-dev.txt"], check=True)


In [ ]:
import torch
print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
if not torch.cuda.is_available():
    print("AVISO: treino em CPU pode ser muito lento. Confirme o runtime GPU.")


## 3. Persistência opcional no Google Drive

A sessão do Colab é efêmera. Monte o Drive para copiar artefatos ao final; o pipeline continua trabalhando em `/content` para melhor desempenho.

In [ ]:
USE_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT = Path("/content/drive/MyDrive/drenavision-artifacts")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)


## 4. Registrar pesquisa e disponibilidade das fontes

In [ ]:
Path("resultados/local").mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable, "ferramentas/scripts/research_datasets.py",
    "--check-access",
    "--output", "resultados/local/adaptation/research_report.json",
], cwd=PROJECT, check=True)


## 5. Baixar fontes aprovadas

O download ocupa centenas de megabytes. Arquivos são verificados por tamanho/checksum quando o catálogo oferece esses dados. CLICK usa HTTP Range para obter somente a amostra 100/20/20.

In [ ]:
subprocess.run([
    sys.executable, "ferramentas/scripts/download_datasets.py",
    "--source", "all",
    "--seed", "42",
    "--click-train", "100",
    "--click-val", "20",
    "--click-test", "20",
], cwd=PROJECT, check=True)


## 6. Preparar o dataset v2 sem sobrescrever v1

A preparação recupera os arquivos brutos pelos IDs/hashes do manifesto v1 versionado; não exige um dataset v1 processado. Exclui keypoints ausentes (0,0,0), usa margem de 10% e mantém os mesmos IDs de validação/teste com caixas corrigidas. O domínio acrescentado ao treino não serve como teste independente. Uma pasta v2 já preenchida exige novo destino e configuração, sem sobrescrever evidências.

In [ ]:
subprocess.run([
    sys.executable, "ferramentas/scripts/prepare_domain_annotations.py",
], cwd=PROJECT, check=True)
subprocess.run([
    sys.executable, "ferramentas/scripts/prepare_adaptation.py",
    "--domain-root", "dados/interim/adaptation",
], cwd=PROJECT, check=True)


## 7. Validar integridade, classes, caixas e vazamento

A opção `--phash` detecta imagens visualmente próximas em splits diferentes. O treino não deve continuar se o comando retornar erro.

In [ ]:
subprocess.run([
    sys.executable, "ferramentas/scripts/validate_dataset.py",
    "--dataset-root", "dados/processed/drenavision_v2",
    "--yaml", "dados/processed/drenavision_v2/dataset.yaml",
    "--manifest", "dados/processed/drenavision_v2/data_manifest.csv",
    "--phash",
    "--phash-threshold", "4",
    "--report", "resultados/local/adaptation/validation_report.json",
], cwd=PROJECT, check=True)


## 8. Inspecionar visualmente uma amostra de anotações

In [ ]:
subprocess.run([
    sys.executable, "ferramentas/scripts/visualize_annotations.py",
    "--dataset-root", "dados/processed/drenavision_v2",
    "--output-root", "resultados/local/adaptation/preview",
    "--samples-per-split", "6",
    "--seed", "42",
], cwd=PROJECT, check=True)

from IPython.display import display
from PIL import Image
preview_paths = sorted(Path("resultados/local/adaptation/preview").rglob("*.jpg"))[:6]
for preview in preview_paths:
    print(preview)
    display(Image.open(preview))


## 9. Executar fine-tuning customizado

O treino completo usa 100 épocas como máximo e early stopping. Para um teste técnico, reduza `EPOCHS`, mas não apresente o checkpoint de smoke test como resultado final.

In [ ]:
EPOCHS = 100  # @param {type:"integer"}
BATCH = 12  # @param {type:"integer"}
DEVICE = "0" if torch.cuda.is_available() else "cpu"
RUN_NAME = "yolo26n_corrected_v2_colab"  # @param {type:"string"}

import shutil
from ultralytics.utils.downloads import attempt_download_asset
models = Path("resultados/local/models")
models.mkdir(parents=True, exist_ok=True)
base = Path(attempt_download_asset(models / "yolo26n.pt"))
shutil.copy2(base, models / "yolo26n-base.pt")

subprocess.run([
    sys.executable, "ferramentas/scripts/train.py",
    "--config", "ferramentas/configs/train_v2.yaml",
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--name", RUN_NAME,
], cwd=PROJECT, check=True)


## 10. Avaliar exclusivamente no teste corrigido v2

Esta célula avalia o candidato gerado nesta sessão; não promove automaticamente o modelo ativo. AP de validação usado para selecionar época não é AP de teste. Para comparar o baseline, obtenha best_v1.pt pela release histórica conforme REPRODUZIR.md e avalie ambos nos mesmos rótulos v2. Valores v1 medidos nas caixas antigas não são diretamente comparáveis.

In [ ]:
EVAL_NAME = "candidate_v2_corrected_test_colab"  # @param {type:"string"}
subprocess.run([
    sys.executable, "ferramentas/scripts/evaluate.py",
    "--config", "ferramentas/configs/evaluate_v2.yaml",
    "--model", "resultados/local/models/candidate_v2.pt",
    "--device", DEVICE,
    "--name", EVAL_NAME,
], cwd=PROJECT, check=True)


In [ ]:
import json
metric_files = sorted(Path("resultados/local/evaluation").rglob("metrics.json"), key=lambda p: p.stat().st_mtime)
if not metric_files:
    raise FileNotFoundError("A avaliação não gerou metrics.json")
metrics_path = metric_files[-1]
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
print("Arquivo autoritativo:", metrics_path)
print(json.dumps(metrics["metrics"], indent=2, ensure_ascii=False))


## 11. Inferência real em uma imagem de teste

In [ ]:
test_images = sorted(Path("dados/processed/drenavision_v2/images/test").glob("*"))
test_images = [p for p in test_images if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
if not test_images:
    raise FileNotFoundError("O split test não contém imagens")
test_image = test_images[0]
subprocess.run([
    sys.executable, "ferramentas/scripts/predict.py",
    "--input", str(test_image),
    "--model", "resultados/local/models/candidate_v2.pt",
    "--output-dir", "resultados/local/predictions",
    "--device", DEVICE,
], cwd=PROJECT, check=True)
annotated = Path("resultados/local/predictions") / f"{test_image.stem}_annotated.jpg"
display(Image.open(annotated))


## 12. Preservar checkpoint, manifests e métricas

O pacote não inclui o dataset bruto. Verifique os hashes e copie o ZIP para armazenamento persistente antes de encerrar a sessão.

In [ ]:
import hashlib
import shutil
import tempfile

checkpoint = Path("resultados/local/models/candidate_v2.pt")
if not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)
sha256 = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
print("candidate_v2.pt SHA-256:", sha256)

bundle_root = Path(tempfile.mkdtemp(prefix="drenavision_evidence_", dir="/content"))
for source in [
    Path("resultados/local/models"),
    Path("resultados/local/training"),
    Path("resultados/local/evaluation"),
    Path("resultados/local/adaptation"),
    Path("dados/processed/drenavision_v2/data_manifest.csv"),
    Path("dados/manifests/research_report.json"),
    Path("dados/manifests"),
    Path("ferramentas/configs"),
]:
    if source.is_dir():
        shutil.copytree(source, bundle_root / source, dirs_exist_ok=True)
    elif source.is_file():
        target = bundle_root / source
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
archive = Path(shutil.make_archive(str(bundle_root), "zip", bundle_root))
print("Pacote:", archive, archive.stat().st_size, "bytes")
if USE_DRIVE:
    shutil.copy2(archive, DRIVE_OUTPUT / archive.name)
    print("Cópia no Drive:", DRIVE_OUTPUT / archive.name)


In [ ]:
DOWNLOAD_NOW = False  # @param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files
    files.download(str(archive))


## Checklist de encerramento

- confira `training_manifest.json` e `metrics.json`;
- registre o commit e o SHA-256 de `candidate_v2.pt`; só promova após avaliação comparativa e revisão;
- salve o ZIP fora da sessão;
- não copie métricas para o relatório sem vinculá-las ao checkpoint/dataset;
- não declare treino concluído se qualquer célula relevante falhou.